# Détection d'intrusions — Log Application
## Projet IDS | Apprentissage non supervisé — Isolation Forest

In [1]:
import pandas as pd
import numpy as np
import re
from datetime import datetime
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import IsolationForest
import warnings
warnings.filterwarnings('ignore')


In [2]:
import os
os.chdir(r"C:\Users\asus\Desktop\IDS")

# Charger le vrai dataset application
df = pd.read_csv("Non supervisé/Data/Application_hpisot-winserve.csv")

print(f"✅ Dataset chargé : {df.shape[0]} lignes, {df.shape[1]} colonnes")
print(f"\nColonnes : {df.columns.tolist()}")
print(f"\nAperçu :")
display(df.head())
print(f"\nDistribution Level :")
print(df['Level'].value_counts() if 'Level' in df.columns else "Colonne Level absente")

✅ Dataset chargé : 7938 lignes, 6 colonnes

Colonnes : ['Level', 'Date and Time', 'Source', 'Event ID', 'Task Category', 'Message ']

Aperçu :


,Level,Date and Time,Source,Event ID,Task Category,Message
0,Information,12/19/2016 3:51:15 PM,Desktop Window Manager,9009,NaN,The Desktop Window Manager has exited with cod...
1,Error,12/19/2016 3:48:39 PM,RSyslogWindowsAgent,151,NaN,Numerous runtime events were encountered .\nTh...
2,Error,12/19/2016 3:48:39 PM,RSyslogWindowsAgent,114,NaN,Error while applying an action - skipping acti...
3,Error,12/19/2016 3:48:38 PM,RSyslogWindowsAgent,114,NaN,Error while applying an action - skipping acti...
4,Error,12/19/2016 3:48:25 PM,RSyslogWindowsAgent,114,NaN,Error while applying an action - skipping acti...



Distribution Level :
Level
Information    7535
Error           403
Name: count, dtype: int64


## 2. Nettoyage des données

In [3]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from datetime import datetime

print("=== NETTOYAGE ===")

# 0. Nettoyer les noms de colonnes (enlever espaces)
df.columns = df.columns.str.strip()
print(f"Colonnes après strip : {df.columns.tolist()}")

# 1. Valeurs manquantes
print(f"\n1. Valeurs manquantes :\n{df.isnull().sum()}")

# 2. Doublons
initial_shape = df.shape
df = df.drop_duplicates()
print(f"\n2. Doublons supprimés : {initial_shape[0] - df.shape[0]}")

# 3. Nettoyage colonnes texte
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].str.strip()

# 4. Nettoyage message
df['Message'] = df['Message'].str.replace('\n', ' ').str.replace('\r', ' ')

# 5. Conversion date
def convert_date(date_str):
    try:
        return datetime.strptime(str(date_str), '%m/%d/%Y %I:%M:%S %p')
    except:
        return None

df['DateTime'] = df['Date and Time'].apply(convert_date)
df['Hour']      = df['DateTime'].dt.hour.fillna(0).astype(int)
df['Minute']    = df['DateTime'].dt.minute.fillna(0).astype(int)
df['DayOfWeek'] = df['DateTime'].dt.dayofweek.fillna(0).astype(int)

# 6. Event ID numérique
df['Event ID'] = pd.to_numeric(df['Event ID'], errors='coerce').fillna(0)

print(f"\n✅ Shape après nettoyage : {df.shape}")

# === PREPROCESSING ===
print("\n=== PREPROCESSING ===")

# Encodage catégorielles
label_encoders = {}
for col in ['Level', 'Source', 'Task Category']:
    if col in df.columns:
        le = LabelEncoder()
        df[f'{col}_encoded'] = le.fit_transform(df[col].astype(str))
        label_encoders[col] = le
        print(f"✓ {col} encodé")

# Features textuelles
df['Message_Length'] = df['Message'].str.len()
df['Has_Error']      = df['Message'].str.contains(
                       'error|Error|ERROR', na=False).astype(int)
df['Has_Fatal']      = df['Message'].str.contains(
                       'fatal|Fatal', na=False).astype(int)

# Sélection features
feature_cols = ['Event ID', 'Hour', 'Minute', 'DayOfWeek',
                'Message_Length', 'Has_Error', 'Has_Fatal',
                'Level_encoded', 'Source_encoded']
feature_cols = [col for col in feature_cols if col in df.columns]

X = df[feature_cols].fillna(0).values

# Normalisation
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"\n✅ X_scaled shape : {X_scaled.shape}")
print(f"✅ Features : {feature_cols}")

=== NETTOYAGE ===
Colonnes après strip : ['Level', 'Date and Time', 'Source', 'Event ID', 'Task Category', 'Message']

1. Valeurs manquantes :
Level               0
Date and Time       0
Source              0
Event ID            0
Task Category    7846
Message             0
dtype: int64

2. Doublons supprimés : 4

✅ Shape après nettoyage : (7934, 10)

=== PREPROCESSING ===
✓ Level encodé
✓ Source encodé
✓ Task Category encodé

✅ X_scaled shape : (7934, 9)
✅ Features : ['Event ID', 'Hour', 'Minute', 'DayOfWeek', 'Message_Length', 'Has_Error', 'Has_Fatal', 'Level_encoded', 'Source_encoded']


## 3. Modèle — Isolation Forest
Détection d'anomalies sans étiquettes. Isolation Forest isole les points
anormaux en les séparant plus rapidement que les points normaux.

In [4]:
# ========================
# 4. MODÈLE ISOLATION FOREST (CORRIGÉ)
# ========================

print("=== MODÈLE ISOLATION FOREST ===")

# 4.1 Entraînement du modèle
# contamination: proportion estimée d'anomalies (ici 20% car plusieurs erreurs réseau)
iso_forest = IsolationForest(
    contamination=0.2,      # 20% d'anomalies attendues
    random_state=42,
    n_estimators=100,
    max_samples='auto'
)

# Prédiction des anomalies
df['Anomaly_Score'] = iso_forest.fit_predict(X_scaled)
df['Anomaly'] = df['Anomaly_Score'].map({1: 'Normal', -1: 'Anomalie'})

# 4.2 Scores de décision (plus négatif = plus anormal)
df['Decision_Score'] = iso_forest.decision_function(X_scaled)

print(f"1. Résultats de la détection:")
print(f"   - Événements normaux: {(df['Anomaly'] == 'Normal').sum()}")
print(f"   - Anomalies détectées: {(df['Anomaly'] == 'Anomalie').sum()}")
print(f"   - Taux d'anomalies: {(df['Anomaly'] == 'Anomalie').sum()/len(df)*100:.1f}%")

# 4.3 Analyse des caractéristiques des anomalies (CORRIGÉ - utilise uniquement des colonnes numériques)
print("\n2. Caractéristiques des anomalies détectées:")
anomalies = df[df['Anomaly'] == 'Anomalie']
normals = df[df['Anomaly'] == 'Normal']

# Sélection des colonnes numériques uniquement
numeric_columns = ['Event ID', 'Hour', 'Minute', 'DayOfWeek', 'Message_Length', 
                   'Message_Word_Count', 'Has_Error_Code', 'Source_Event_Freq', 
                   'Seconds_Since_Start']

for col in numeric_columns:
    if col in df.columns:
        mean_normal = normals[col].mean() if len(normals) > 0 else 'N/A'
        mean_anomaly = anomalies[col].mean() if len(anomalies) > 0 else 'N/A'
        print(f"   - {col}:")
        print(f"     Normale moyenne: {mean_normal:.2f}" if isinstance(mean_normal, (int, float)) else f"     Normale moyenne: {mean_normal}")
        print(f"     Anomalie moyenne: {mean_anomaly:.2f}" if isinstance(mean_anomaly, (int, float)) else f"     Anomalie moyenne: {mean_anomaly}")

# Analyse des colonnes catégorielles séparément
print("\n3. Analyse des colonnes catégorielles:")
for col in ['Level', 'Source', 'Task Category']:
    if col in df.columns:
        print(f"   - {col}:")
        print(f"     Distribution normale: {normals[col].value_counts().to_dict() if len(normals)>0 else {}}")
        print(f"     Distribution anomalie: {anomalies[col].value_counts().to_dict() if len(anomalies)>0 else {}}")

print("\n" + "="*50 + "\n")

=== MODÈLE ISOLATION FOREST ===
1. Résultats de la détection:
   - Événements normaux: 6350
   - Anomalies détectées: 1584
   - Taux d'anomalies: 20.0%

2. Caractéristiques des anomalies détectées:
   - Event ID:
     Normale moyenne: 435.25
     Anomalie moyenne: 5793.23
   - Hour:
     Normale moyenne: 13.68
     Anomalie moyenne: 12.37
   - Minute:
     Normale moyenne: 27.85
     Anomalie moyenne: 27.97
   - DayOfWeek:
     Normale moyenne: 3.33
     Anomalie moyenne: 2.28
   - Message_Length:
     Normale moyenne: 963.04
     Anomalie moyenne: 1502.42

3. Analyse des colonnes catégorielles:
   - Level:
     Distribution normale: {'Information': 6350}
     Distribution anomalie: {'Information': 1182, 'Error': 402}
   - Source:
     Distribution normale: {'Windows PowerShell (logit.ps)': 3543, 'Microsoft-Windows-Security-SPP': 2775, 'Microsoft-Windows-User Profiles Service': 10, 'MsiInstaller': 6, 'Microsoft-Windows-WMI': 5, 'cloudbase-init': 4, 'RSyslogWindowsAgent': 4, 'Microsoft-

## 4. Résultats et visualisation

In [5]:
# DataFrame de sortie avec les 10 premières lignes
df[['Level', 'Source', 'Event ID', 'Message', 'Anomaly', 'Anomaly_Score', 'Decision_Score']].head(10)

,Level,Source,Event ID,Message,Anomaly,Anomaly_Score,Decision_Score
0,Information,Desktop Window Manager,9009,The Desktop Window Manager has exited with cod...,Anomalie,-1,-0.114518
1,Error,RSyslogWindowsAgent,151,Numerous runtime events were encountered . Thi...,Anomalie,-1,-0.136921
2,Error,RSyslogWindowsAgent,114,Error while applying an action - skipping acti...,Anomalie,-1,-0.115445
3,Error,RSyslogWindowsAgent,114,Error while applying an action - skipping acti...,Anomalie,-1,-0.115445
4,Error,RSyslogWindowsAgent,114,Error while applying an action - skipping acti...,Anomalie,-1,-0.115445
5,Error,RSyslogWindowsAgent,114,Error while applying an action - skipping acti...,Anomalie,-1,-0.115445
6,Error,RSyslogWindowsAgent,114,Error while applying an action - skipping acti...,Anomalie,-1,-0.115445
7,Error,RSyslogWindowsAgent,114,Error while applying an action - skipping acti...,Anomalie,-1,-0.115445
8,Error,RSyslogWindowsAgent,114,Error while applying an action - skipping acti...,Anomalie,-1,-0.115445
9,Error,RSyslogWindowsAgent,114,Error while applying an action - skipping acti...,Anomalie,-1,-0.115445


In [6]:
import pandas as pd, os

base = r"C:\Users\asus\Desktop\IDS\Non supervisé\Notebooks\outputs"
os.makedirs(base, exist_ok=True)

df_app_normalized = pd.DataFrame({
    "timestamp"     : df["Date and Time"].values
                      if "Date and Time" in df.columns else "unknown",
    "source_ip"     : "unknown",   # log applicatif — pas d'IP
    "dest_ip"       : "unknown",
    "log_source"    : "application",
    "source_name"   : df["Source"].values
                      if "Source" in df.columns else "unknown",
    "event_id"      : df["Event ID"].values
                      if "Event ID" in df.columns else "unknown",
    "anomaly_score" : iso_forest.decision_function(X_scaled),
    "prediction"    : df["Anomaly"].values,
    "alert_level"   : ["HIGH" if a == "Anomalie" else "NORMAL"
                       for a in df["Anomaly"].values],
})

df_app_normalized.to_csv(
    os.path.join(base, "application_output_normalized.csv"), index=False
)
print("✅ application_output_normalized.csv sauvegardé")
print(f"   {len(df_app_normalized)} lignes")
print(df_app_normalized.head())

✅ application_output_normalized.csv sauvegardé
   7934 lignes
               timestamp source_ip  dest_ip   log_source  \
0  12/19/2016 3:51:15 PM   unknown  unknown  application   
1  12/19/2016 3:48:39 PM   unknown  unknown  application   
2  12/19/2016 3:48:39 PM   unknown  unknown  application   
3  12/19/2016 3:48:38 PM   unknown  unknown  application   
4  12/19/2016 3:48:25 PM   unknown  unknown  application   

              source_name  event_id  anomaly_score prediction alert_level  
0  Desktop Window Manager      9009      -0.114518   Anomalie        HIGH  
1     RSyslogWindowsAgent       151      -0.136921   Anomalie        HIGH  
2     RSyslogWindowsAgent       114      -0.115445   Anomalie        HIGH  
3     RSyslogWindowsAgent       114      -0.115445   Anomalie        HIGH  
4     RSyslogWindowsAgent       114      -0.115445   Anomalie        HIGH  


In [7]:
import joblib, os
base = r"C:\Users\asus\Desktop\IDS\models"
os.makedirs(base, exist_ok=True)
joblib.dump(iso_forest, base + r"\app_isolation_forest.pkl")
joblib.dump(scaler,     base + r"\app_scaler.pkl")
print("✅ application model sauvegardé")

✅ application model sauvegardé
